# Stage C / NB 16 — decoding, prompt, and reasoning-depth sensitivity (E6)

Protocol reference: experiment family **E6**; referee point **R1.1**. Declared **exploratory**
in the statistics plan — these arms are not part of the confirmatory families and carry no
multiplicity correction, because their job is to test whether the headline result is an
artefact, not to add a result of their own.

## What this notebook is for

NB 15 reports one number per arm, produced under greedy decoding with one prompt. A referee is
entitled to ask whether that number survives a different temperature, a rephrased instruction,
or a request for more reasoning. Four families answer that:

| family | varies | the question it settles |
| --- | --- | --- |
| E6-D | temperature x top-p | is the result an artefact of a lucky decoding setting? |
| E6-C | self-consistency n in {1, 3, 5} | does sampling and voting buy accuracy, and at what latency? |
| E6-P | 3 prompt variants x 3 paraphrases | how brittle is the system to wording? |
| E6-T | reasoning depth | does asking for more reasoning help, and what does it cost? |

Accuracy is reported **beside latency and token count** throughout. An arm that gains 0.1 MAE
for five times the compute is not an improvement, and the table should make that obvious rather
than leaving it to the discussion.

## Where it runs, and why not on test folds

E6 runs on the **same evaluation half of fold 0's inner-validation pool** that NB 15 used to
select its reasoner and setting. Sensitivity analysis is a form of exploration: running 30-odd
configurations against held-out folds and then reporting the headline arm from the same data
would quietly turn the test set into a development set. Nothing here touches a test fold.

## E6-Q is a precondition, not an arm

Before any of this means anything, the model must emit **exactly one** JSON object per answer.
The tested notebooks emitted the object and then repeated it, plus a stray `<unused94>` control
token. Parsing recovered the first copy, so the metrics were valid and nobody noticed — but
token counts, latency, and every self-consistency result would have been measuring repetition.
Section 4 verifies the fix before any grid runs, and the gate blocks on it.

## Outputs (under `stage_C/nb16_sensitivity/`)
`e6_sensitivity_grid.csv`, `prompt_brittleness.csv`, `latency_vs_accuracy.csv`,
`e6q_output_hygiene.csv`, `gate_nb16.json`.

**Interruption safety.** As in NB 15: one journal per configuration, one line per image,
fingerprinted on everything that changes the answer. Re-run from the top; finished
configurations are skipped and a partial one resumes at the next image.

## 1. Imports and the Stage A path contract

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import re
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions live with the Stage B notebooks; every arm in Table 2 and every
# fusion/reasoner arm here must be scored by identical code or the comparison is invalid.
_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_B",
           Path.cwd().parent.parent / "notebooks" / "stage_B"]
for _candidate in _SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(f"cxr_metrics.py not found. Searched: {_SEARCH}")
import cxr_metrics as cm

# Stage C's own shared module (prompt rendering, JSON parsing, journals). Kept beside the
# notebooks for the same reason cxr_metrics.py is: two notebooks parsing model output slightly
# differently would show up as a metric difference nobody could explain.
for _candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_C",
                   Path.cwd().parent.parent / "notebooks" / "stage_C"]:
    if (_candidate / "stage_c_reasoner.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

FALLBACK_STAGE_A = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
for _candidate in [FALLBACK_STAGE_A / "nb00_environment" / "stage_a_paths.json",
                   Path.cwd() / "stage_a_paths.json",
                   Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json"]:
    if _candidate.is_file():
        stage_paths = json.loads(_candidate.read_text(encoding="utf-8"))
        print("Path contract:", _candidate)
        break
else:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
STAGE_C_DIR = STAGE_ROOT / "stage_C"
STAGE_C_DIR.mkdir(parents=True, exist_ok=True)
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # identical to Stage B, so inner splits match exactly

print("Stage C output:", STAGE_C_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
import stage_c_reasoner as scr

print("stage_c_reasoner loaded from", scr.__file__)

## 2. Configuration — inherited from NB 15, not re-chosen

The reasoner and the E3 setting come from NB 15's `run_config.json`. Re-picking them here would
mean the sensitivity study describes a system that was never evaluated.

This is also the exact place NB 10 broke: it was built by transforming NB 09's cells, and a
bulk rename rewrote the line that was supposed to **read** NB 09's artifact. The load below
therefore distinguishes "NB 15 has not run yet" from "the path is wrong" — the remedies are
hours apart.

In [ ]:
NB16_DIR = STAGE_C_DIR / "nb16_sensitivity"
JOURNAL_DIR = NB16_DIR / "journals"
for d in (NB16_DIR, JOURNAL_DIR):
    d.mkdir(parents=True, exist_ok=True)
NB13_DIR = STAGE_C_DIR / "nb13_registry"
NB15_DIR = STAGE_C_DIR / "nb15_reasoner"        # READ ONLY -- NB 15's outputs

config_path = NB15_DIR / "run_config.json"
if not config_path.is_file():
    raise FileNotFoundError(
        f"{config_path} not found.\n"
        f"  If {NB15_DIR} exists but is empty, NB 15 has not finished — run it first.\n"
        f"  If {NB15_DIR} does not exist at all, the Stage C root is wrong: STAGE_C_DIR "
        f"resolved to {STAGE_C_DIR}, which should contain nb13_registry/, nb14_fusion/ and "
        "nb15_reasoner/. Check stage_a_paths.json rather than editing this path.")
nb15_config = json.loads(config_path.read_text(encoding="utf-8"))

REASONER = nb15_config["reasoner_selected"]
SETTING = nb15_config["setting_selected"]
ROSTER = nb15_config["roster_full"]
SELECTION_FOLD = nb15_config.get("selection_fold", 0)
print(f"Inherited from NB 15: reasoner={REASONER}  setting={SETTING}  roster={ROSTER}")
print(f"  full-roster arm was {nb15_config.get('full_roster_arm')}, "
      f"denominator {nb15_config.get('framework_denominator')} images")

# The candidate specs are re-declared rather than imported, but the one that matters is
# asserted against NB 15's record below.
REASONER_CANDIDATES = OrderedDict([
    ("R_medgemma_base", {"model_id": "google/medgemma-1.5-4b-it",
                         "loader": "image_text_to_text", "adapter": None}),
    ("R_medgemma_lora", {"model_id": "google/medgemma-1.5-4b-it",
                         "loader": "image_text_to_text",
                         "adapter": str(STAGE_B_DIR / "nb09_medgemma_lora" / "folds"
                                        / "fold_{fold}" / "best_adapter")}),
    ("R_qwen_base",     {"model_id": "Qwen/Qwen3.5-4B", "loader": "multimodal_lm",
                         "adapter": None, "disable_thinking": True}),
    ("R_qwen_lora",     {"model_id": "Qwen/Qwen3.5-4B", "loader": "multimodal_lm",
                         "adapter": str(STAGE_B_DIR / "nb10_qwen_lora" / "folds"
                                        / "fold_{fold}" / "best_adapter"),
                         "disable_thinking": True}),
    ("R_nvreason",      {"model_id": "nvidia/NV-Reason-CXR-3B", "loader": "auto",
                         "adapter": None}),
])
if REASONER not in REASONER_CANDIDATES:
    raise RuntimeError(f"NB 15 selected {REASONER}, which is not in this notebook's candidate "
                       "list. The two notebooks disagree about what the system is.")
SPEC = {**REASONER_CANDIDATES[REASONER], "_label": REASONER}
recorded_spec = nb15_config.get("reasoner_spec")
if not recorded_spec:
    raise RuntimeError("NB 15 run_config predates model-spec recording; rerun revised NB 15.")
if recorded_spec != REASONER_CANDIDATES[REASONER]:
    raise RuntimeError(f"Reasoner spec drift: NB 15 recorded {recorded_spec}, but NB 16 "
                       f"would load {REASONER_CANDIDATES[REASONER]}.")

# ---- E6-D: temperature x top-p ------------------------------------------------------------
TEMPERATURES = [0.0, 0.2, 0.7, 1.0]
TOP_P = [0.8, 0.95, 1.0]
# ---- E6-C: self-consistency ---------------------------------------------------------------
SELF_CONSISTENCY_N = [1, 3, 5]
SELF_CONSISTENCY_TEMPERATURE = 0.7
# ---- E6-T: reasoning depth ----------------------------------------------------------------
DEPTHS = ["none", "short", "full"]

MAX_NEW_TOKENS = 384
MAX_NEW_TOKENS_FULL_TRACE = 768     # the "full stepwise trace" arm needs room to finish
USE_JSON_STOP_CRITERIA = True
E6_MAX_IMAGES = 150                 # per configuration; E6 is exploratory, not confirmatory
N_BOOTSTRAP = 2000

RUN_E6D, RUN_E6C, RUN_E6P, RUN_E6T = True, True, True, True
SMOKE_TEST = False
if SMOKE_TEST:
    E6_MAX_IMAGES = 8

FORCE_RECOMPUTE_CONFIGS = []
FORCE_RECOMPUTE_ALL = False
MAX_SESSION_HOURS = 35.0      # Biowulf limit is 36 h; stop cleanly between images
SESSION_DEADLINE = scr.make_session_deadline(MAX_SESSION_HOURS)

print()
print(f"E6-D grid : {len(TEMPERATURES)} temperatures x {len(TOP_P)} top-p")
print(f"E6-C      : n in {SELF_CONSISTENCY_N} at T={SELF_CONSISTENCY_TEMPERATURE}")
print(f"E6-T      : depths {DEPTHS}")
print(f"Soft stop : {MAX_SESSION_HOURS:.1f} h after this cell")
print("Output    :", NB16_DIR)

## 3. The evaluation rows — the same ones NB 15 selected on

Rebuilt from the registry with NB 13's fold-relative membership and NB 15's split rule and
seeds, so the images are literally the same. Reproducing the split is safer than reading a
saved list: if either rule ever changes, this breaks loudly instead of quietly comparing
against a different sample.

In [ ]:
registry_path = NB13_DIR / "agent_registry.parquet"
if registry_path.is_file():
    try:
        registry = pd.read_parquet(registry_path)
    except Exception:
        registry = pd.read_csv(NB13_DIR / "agent_registry.csv")
else:
    registry = pd.read_csv(NB13_DIR / "agent_registry.csv")

INNER_FOLD_COLUMNS = [f"inner_fold_{k}" for k in range(N_FOLDS)]
selection_column = f"selected_for_fold_{SELECTION_FOLD}"
if selection_column not in registry.columns:
    raise RuntimeError(f"{selection_column} missing; rerun revised NB 13.")
AGENTS = [a for a in ["A1", "A2", "A3", "A4", "A5", "A6"]
          if a in set(registry["agent"])]
selected = registry[(registry[selection_column] == True)
                    & registry["agent"].isin(AGENTS)].copy()

wide = selected.pivot_table(index="image_key", columns="agent", values="mrale_total",
                            aggfunc="first").reindex(columns=AGENTS)
unc = selected.pivot_table(index="image_key", columns="agent", values="mrale_uncertainty",
                           aggfunc="first")
covid_pred = selected.pivot_table(index="image_key", columns="agent", values="covid_pred",
                                  aggfunc="first")
covid_score = selected.pivot_table(index="image_key", columns="agent", values="covid_score",
                                   aggfunc="first")
meta = (registry.drop_duplicates("image_key").set_index("image_key")
        [["fold", "group_id", "gt_mrale_total", "gt_covid", "severity_band"]
         + INNER_FOLD_COLUMNS])
presence = selected.pivot_table(index="image_key", columns="agent", values="arm",
                                aggfunc="size", fill_value=0)
for agent in AGENTS:
    if agent not in presence.columns:
        presence[agent] = 0
eligible = presence.index[(presence[AGENTS] > 0).all(axis=1)]
frame = meta.join(wide, how="inner").loc[eligible]

views = pd.read_csv(NB04_DIR / "view_index.csv").set_index("image_key")
def preferred_image_path(record):
    for column in ["v0_image", "v1_thorax_image"]:
        value = record.get(column)
        if isinstance(value, str) and value.strip() and Path(value).is_file():
            return value
    return None

IMAGE_PATHS = {str(k): preferred_image_path(r)
               for k, r in views.to_dict("index").items()}
missing_paths = [k for k in frame.index if not IMAGE_PATHS.get(str(k))]
if missing_paths:
    raise FileNotFoundError(
        f"{len(missing_paths):,} framework images have no readable V0/V1 path (examples: "
        f"{missing_paths[:5]}). NB 16 cannot silently change NB 15's denominator.")
evidence = defaultdict(dict)
for label, path in [("A6", STAGE_B_DIR / "nb08_biomedclip_entity" / "entity_findings.jsonl"),
                    ("A4", STAGE_B_DIR / "nb11_nvreason" / "nvreason_findings.jsonl")]:
    if path.is_file():
        for row in cm.read_jsonl(path):
            evidence[str(row.get("image_key"))][label] = row

# NB 15's nested selection split, reproduced exactly.
pool = frame[frame[f"inner_fold_{SELECTION_FOLD}"] == 1]
pool_groups = sorted(pool["group_id"].astype(str).unique())
rng = random.Random(SEED + 9001)
rng.shuffle(pool_groups)
half = len(pool_groups) // 2
fit_groups, eval_groups = set(pool_groups[:half]), set(pool_groups[half:])
FIT_KEYS = [str(k) for k in pool.index[pool["group_id"].astype(str).isin(fit_groups)]]
EVAL = pool[pool["group_id"].astype(str).isin(eval_groups)].sort_index()

# NB 15 capped its evaluation half before scoring. That cap has to be reproduced BEFORE this
# notebook applies its own, or E6's rows are drawn from the wider pool and end up barely
# overlapping the rows NB 15 actually scored -- which would silently defeat both the prompt
# byte-identity check and the reference-arm reproduction check in the gate.
nb15_cap = nb15_config.get("selection_max_images")
if nb15_cap and len(EVAL) > nb15_cap:
    EVAL = EVAL.sample(n=nb15_cap, random_state=SEED).sort_index()

expected = nb15_config.get("selection_eval_images")
if expected and len(EVAL) != expected:
    raise RuntimeError(
        f"Reproduced {len(EVAL)} evaluation images but NB 15 recorded {expected}. The split "
        "rule, the seed or the cap has changed, so this study would not be measuring NB 15's "
        "system. Do not adjust the numbers to match — find out which rule moved.")

if E6_MAX_IMAGES and len(EVAL) > E6_MAX_IMAGES:
    EVAL = EVAL.sample(n=E6_MAX_IMAGES, random_state=SEED).sort_index()

print(f"Agents: {AGENTS}")
print(f"Selection pool (fold {SELECTION_FOLD} inner validation): {len(pool):,} images")
print(f"  reliability half: {len(FIT_KEYS):,}   evaluation rows: {len(EVAL):,} "
      f"(NB 15 cap {nb15_cap}, then E6 cap {E6_MAX_IMAGES}) — a subset of NB 15's rows")
overlap = set(FIT_KEYS) & set(map(str, EVAL.index))
if overlap:
    raise RuntimeError(f"{len(overlap)} images are in both halves of the selection split.")
print("  halves are disjoint by image and by patient group — verified")

## 4. Prompts — the reference, three variants, three paraphrases, three depths

The **reference prompt is NB 15's**, rebuilt here rather than imported. That is a duplication
risk, so it is turned into an assertion: section 5 recomputes the prompt hash for images NB 15
already scored under the same setting and requires an exact match. If the two notebooks have
drifted by so much as a space, this notebook refuses to run rather than reporting a sensitivity
study of a prompt that was never used.

**E6-P variants** (protocol: terse schema-only / rubric-explicit / rubric + worked example):

- `P1_terse` — schema and nothing else. Tests how much the rubric is carrying.
- `P2_rubric` — the reference, with the extent/density definitions spelled out.
- `P3_rubric_example` — the rubric plus one worked in-context example.

Each has three paraphrases. Paraphrases say the same thing in different words, so the spread
across them is **prompt brittleness**: a system whose MAE moves by more than its confidence
interval when the wording changes is not measuring the radiograph.

The paraphrases are pre-registered here in code, not generated on the fly, so they cannot be
tuned after seeing results.

**E6-T depths.** `none` removes the rationale field, `short` is the reference (<= 30 words),
`full` asks for an explicit list of reasoning steps inside the JSON. Only auditable evidence
summaries are published; private chain-of-thought is not reproduced in the paper.

In [ ]:
AGENT_DESCRIPTION = {
    "A1": "anatomy-aware per-lung reader (localises each lung, then scores it)",
    "A2": "MedGemma-1.5-4B fine-tuned with LoRA on this cohort",
    "A3": "Qwen3.5-4B fine-tuned with LoRA on this cohort",
    "A4": "NV-Reason-CXR-3B, a reasoning model that reports findings and may abstain",
    "A5": "CXformer frozen encoder with an ordinal probe",
    "A6": "BiomedCLIP contrastive entity scorer over 22 radiographic findings",
    "E0g": "conventional CNN/ViT classifier trained end-to-end",
}

TERSE_SYSTEM = ("You score chest radiographs for mRALE (0-24) and COVID-19. "
                "Answer ONLY with a single JSON object and nothing else.")

WORKED_EXAMPLE = (
    "Worked example. A radiograph with hazy opacity across about a third of the right lung and "
    "a clear left lung scores extent_right 2, density_right 1, so mrale_right 2; extent_left 0, "
    "density_left 0, so mrale_left 0; mrale_total 2.")

# --- E6-T: what the schema asks for at each reasoning depth --------------------------------
DEPTH_SCHEMA = {
    "none": ('{"extent_right": int 0-4, "density_right": int 0-3, "extent_left": int 0-4, '
             '"density_left": int 0-3, "mrale_right": int 0-12, "mrale_left": int 0-12, '
             '"mrale_total": int 0-24, "covid_positive": "Yes" or "No", '
             '"covid_confidence": float 0-1 for the stated call, '
             '"agents_used": list of agent names you relied on}'),
    "short": scr.DEFAULT_RESPONSE_SCHEMA,
    "full": ('{"reasoning_steps": list of at most 6 short strings, one per observation, '
             '"extent_right": int 0-4, "density_right": int 0-3, "extent_left": int 0-4, '
             '"density_left": int 0-3, "mrale_right": int 0-12, "mrale_left": int 0-12, '
             '"mrale_total": int 0-24, "covid_positive": "Yes" or "No", '
             '"covid_confidence": float 0-1 for the stated call, '
             '"agents_used": list of agent names you relied on, '
             '"rationale": string of at most 60 words}'),
}

# --- E6-P: three variants, three pre-registered paraphrases each ----------------------------
PROMPT_VARIANTS = OrderedDict([
    ("P1_terse", {
        "system": TERSE_SYSTEM,
        "lead": ["Agents have read this radiograph. Their reports follow. Give one final "
                 "answer.",
                 "Automated readers produced the reports below. Return a single final answer.",
                 "Below are agent reports for this image. Produce one answer."],
        "solo": ["Score this radiograph.",
                 "Give the mRALE score and COVID call for this radiograph.",
                 "Read this radiograph and report its scores."],
        "example": False,
    }),
    ("P2_rubric", {                       # the reference variant; paraphrase 0 is NB 15's
        "system": scr.DEFAULT_SYSTEM_PROMPT,
        "lead": ["Independent automated agents have already read this radiograph. Their "
                 "reports are below. Look at the image yourself, then produce ONE final "
                 "answer. You may override any agent; if you do, say so in the rationale.",
                 "Several automated readers have already assessed this radiograph, and their "
                 "reports appear below. Examine the image yourself before giving a single "
                 "final answer. You are free to disagree with any agent; note it if you do.",
                 "The reports below come from independent automated readers of this "
                 "radiograph. Form your own view from the image, then give one final answer, "
                 "mentioning any agent you overrule."],
        "solo": ["Score this radiograph yourself, with no agent assistance.",
                 "Assess this radiograph on your own; no agent reports are provided.",
                 "No agent reports are available. Judge this radiograph yourself."],
        "example": False,
    }),
    ("P3_rubric_example", {
        "system": scr.DEFAULT_SYSTEM_PROMPT,
        "lead": ["Independent automated agents have already read this radiograph. Their "
                 "reports are below. Look at the image yourself, then produce ONE final "
                 "answer. You may override any agent; if you do, say so in the rationale.",
                 "Several automated readers have already assessed this radiograph, and their "
                 "reports appear below. Examine the image yourself before giving a single "
                 "final answer. You are free to disagree with any agent; note it if you do.",
                 "The reports below come from independent automated readers of this "
                 "radiograph. Form your own view from the image, then give one final answer, "
                 "mentioning any agent you overrule."],
        "solo": ["Score this radiograph yourself, with no agent assistance.",
                 "Assess this radiograph on your own; no agent reports are provided.",
                 "No agent reports are available. Judge this radiograph yourself."],
        "example": True,
    }),
])
REFERENCE_VARIANT, REFERENCE_PARAPHRASE, REFERENCE_DEPTH = "P2_rubric", 0, "short"

print(f"E6-P: {len(PROMPT_VARIANTS)} variants x 3 paraphrases = "
      f"{len(PROMPT_VARIANTS) * 3} prompts")
print(f"Reference (must reproduce NB 15 exactly): {REFERENCE_VARIANT} / paraphrase "
      f"{REFERENCE_PARAPHRASE} / depth {REFERENCE_DEPTH}")

## 5. Reliability context, the evidence block, and the byte-identity check

The reliability numbers are recomputed from the same half of the selection pool NB 15 used, so
the evidence block carries identical figures. Then the reference prompt is hashed and compared
against the hashes NB 15 recorded in its own journal for the same images.

This check is the whole reason the duplication is acceptable. Without it, "NB 16 uses NB 15's
prompt" would be a claim in a comment.

In [ ]:
def compute_reliability(keys):
    keys = set(map(str, keys))
    subset = selected[selected["image_key"].astype(str).isin(keys)]
    out = {}
    for agent, group in subset.groupby("agent"):
        rows = group.to_dict("records")
        entry = {"n": len(rows)}
        mrale_rows = [{"gt_mrale_total": r["gt_mrale_total"], "mrale_total": r["mrale_total"]}
                      for r in rows if r.get("gt_mrale_total") is not None]
        if mrale_rows:
            m = cm.mrale_metrics(mrale_rows)
            entry["mrale_mae"] = float(m["mae"])
            entry["mrale_qwk"] = float(m.get("qwk", float("nan")))
        covid_rows = [r for r in rows if r.get("gt_covid") in {"Yes", "No"}]
        if covid_rows:
            c = cm.classification_metrics(
                [r["gt_covid"] for r in covid_rows], [r.get("covid_pred") for r in covid_rows],
                [r.get("covid_score") for r in covid_rows])
            if c.get("auroc") is not None:
                entry["covid_auroc"] = float(c["auroc"])
            scored = [(1 if r["gt_covid"] == "Yes" else 0, r.get("covid_score"))
                      for r in covid_rows if r.get("covid_score") is not None]
            if len(scored) > 30 and len({y for y, _ in scored}) == 2:
                probability = np.clip(np.asarray([p for _, p in scored], dtype=float),
                                      1e-6, 1 - 1e-6)
                logit = np.log(probability / (1 - probability)).reshape(-1, 1)
                if logit.std() > 1e-9:
                    from sklearn.linear_model import LogisticRegression
                    y = np.asarray([y for y, _ in scored], dtype=int)
                    try:
                        calibration = LogisticRegression(penalty=None, max_iter=2000).fit(logit, y)
                    except (TypeError, ValueError):
                        calibration = LogisticRegression(penalty="none", max_iter=2000).fit(logit, y)
                    entry["covid_calibration_slope"] = float(calibration.coef_[0, 0])
        out[agent] = entry
    return out


def _finite(value):
    return value is not None and not (isinstance(value, float) and math.isnan(value))


RELIABILITY = compute_reliability(FIT_KEYS)
spreads = frame.loc[FIT_KEYS, AGENTS].astype(float).std(axis=1, skipna=True) \
    if FIT_KEYS else pd.Series(dtype=float)
CONFLICT_THRESHOLD = float(np.nanquantile(spreads, 0.90)) if len(spreads) >= 20 \
    else float("inf")
CONTEXT = {"label": "selection", "reliability": RELIABILITY,
           "conflict_threshold": CONFLICT_THRESHOLD, "fold": SELECTION_FOLD}
print("Reliability (inner-validation half):",
      {a: round(v["mrale_mae"], 3) for a, v in RELIABILITY.items() if "mrale_mae" in v})


def agent_lines(image_key, roster, setting, ctx):
    """Render the same evidence facts and ordering used by NB 15."""
    lines, provenance = [], {}
    row = frame.loc[image_key]
    for agent in roster:
        matches = selected[(selected["image_key"].astype(str) == str(image_key))
                           & (selected["agent"] == agent)]
        source = matches.iloc[0].to_dict() if len(matches) else {}
        value = row.get(agent)
        entry, facts = {}, []
        if _finite(value):
            entry["mrale_total"] = float(value)
            facts.append(f"mRALE {float(value):.0f}")
            right, left = source.get("mrale_right"), source.get("mrale_left")
            if _finite(right) and _finite(left):
                facts.append(f"right {float(right):.0f}, left {float(left):.0f}")
                entry.update({"mrale_right": float(right), "mrale_left": float(left)})
        rel = ctx["reliability"].get(agent, {})
        # E3b = reliability only; E3c = case uncertainty only; E3d/e combine both.
        if setting in {"E3b", "E3d", "E3e"}:
            bits = []
            if _finite(rel.get("mrale_mae")):
                bits.append(f"validation MAE {rel['mrale_mae']:.2f}")
                entry["shown_mae"] = round(float(rel["mrale_mae"]), 4)
            if _finite(rel.get("covid_auroc")):
                bits.append(f"validation AUROC {rel['covid_auroc']:.3f}")
                entry["shown_auroc"] = round(float(rel["covid_auroc"]), 4)
            if _finite(rel.get("covid_calibration_slope")):
                bits.append(f"calibration slope {rel['covid_calibration_slope']:.2f}")
                entry["shown_calibration_slope"] = round(
                    float(rel["covid_calibration_slope"]), 4)
            if bits:
                facts.append("reliability: " + "; ".join(bits))
        if setting in {"E3c", "E3d", "E3e"} and agent in unc.columns \
                and image_key in unc.index:
            uncertainty_value = unc.at[image_key, agent]
            if _finite(uncertainty_value):
                facts.append(f"case uncertainty {float(uncertainty_value):.2f}")
                entry["case_uncertainty"] = round(float(uncertainty_value), 4)
        if agent in covid_pred.columns and image_key in covid_pred.index:
            call = covid_pred.at[image_key, agent]
            if isinstance(call, str) and call in {"Yes", "No"}:
                score = covid_score.at[image_key, agent] if (agent in covid_score.columns
                                                           and image_key in covid_score.index) else None
                facts.append(f"COVID {call}" + (f" (P+ {float(score):.3f})"
                                                   if _finite(score) else ""))
                entry["covid_pred"] = call
                if _finite(score):
                    entry["covid_score"] = round(float(score), 6)
        finding_row = evidence.get(str(image_key), {}).get(agent, {})
        findings = finding_row.get("findings") or []
        if isinstance(findings, str):
            findings = [findings]
        findings = [str(item) for item in findings[:8]]
        if findings:
            facts.append("findings: " + ", ".join(findings))
            entry["findings"] = findings
        if not facts:
            facts.append("no usable numeric output; agent abstained")
            entry["abstained"] = True
        lines.append(f"- {agent} ({AGENT_DESCRIPTION.get(agent, 'agent')}): "
                     + "; ".join(facts))
        provenance[agent] = entry
    return lines, provenance


def evidence_block(image_key, roster, setting, ctx):
    """Byte-identical to NB 15's block; section 5's hash check enforces that."""
    lines, provenance = agent_lines(image_key, roster, setting, ctx)

    meta = {"agents": provenance, "setting": setting, "roster": list(roster),
            "context": ctx["label"]}
    if not lines:
        return ("No agent outputs are available for this image; judge from the radiograph "
                "alone."), meta

    parts = ["Agent reports for this radiograph:"] + lines
    if setting in {"E3d", "E3e"}:
        ranked = [a for a in roster if _finite(ctx["reliability"].get(a, {}).get("mrale_mae"))]
        ranked.sort(key=lambda a: ctx["reliability"][a]["mrale_mae"])
        if ranked:
            parts += ["",
                      "Reliability ranking on held-out validation data, most accurate first: "
                      + " > ".join(ranked) + ".",
                      "Weight the agents accordingly. Where they disagree, prefer the more "
                      "reliable one unless the image clearly contradicts it."]
            meta["reliability_ranking"] = ranked
    if setting == "E3e":
        values = [v["mrale_total"] for v in provenance.values()
                  if _finite(v.get("mrale_total"))]
        spread = float(np.std(values)) if len(values) > 1 else 0.0
        meta["agent_spread"] = round(spread, 4)
        meta["conflict_flagged"] = bool(spread >= ctx["conflict_threshold"])
        if meta["conflict_flagged"]:
            parts += ["",
                      f"CONFLICT: the agents disagree unusually strongly here (spread "
                      f"{spread:.1f}, above the 90% validation percentile). Re-read the image "
                      "yourself rather than averaging, and say in your rationale which agent "
                      "you sided with."]
    return "\n".join(parts), meta


def build_prompt(image_key, variant, paraphrase, depth, roster=None, setting=None):
    roster = ROSTER if roster is None else roster
    setting = SETTING if setting is None else setting
    spec = PROMPT_VARIANTS[variant]
    schema = DEPTH_SCHEMA[depth]
    if roster:
        block, meta = evidence_block(image_key, roster, setting, CONTEXT)
        lead = spec["lead"][paraphrase]
        example = f"\n\n{WORKED_EXAMPLE}" if spec["example"] else ""
        prompt = f"{lead}\n\n{block}{example}\n\nReturn exactly this JSON:\n{schema}"
    else:
        meta = {"agents": {}, "setting": setting, "roster": [], "context": CONTEXT["label"]}
        example = f"\n\n{WORKED_EXAMPLE}" if spec["example"] else ""
        prompt = f"{spec['solo'][paraphrase]}{example}\n\nReturn exactly this JSON:\n{schema}"
    meta.update({"variant": variant, "paraphrase": paraphrase, "depth": depth,
                 "prompt_sha256": hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]})
    return prompt, spec["system"], meta


# ---- The byte-identity check --------------------------------------------------------------
nb15_journal = NB15_DIR / "journals" / f"E3_{SETTING}__fold{SELECTION_FOLD}.jsonl"
identity_checked, identity_mismatch = 0, []
if nb15_journal.is_file():
    for row in cm.read_jsonl(nb15_journal):
        key = str(row.get("image_key"))
        if key not in frame.index:
            continue
        recorded = ((row.get("extra") or {}).get("provenance") or {}).get("prompt_sha256")
        if not recorded:
            continue
        _, _, meta = build_prompt(key, REFERENCE_VARIANT, REFERENCE_PARAPHRASE,
                                  REFERENCE_DEPTH)
        identity_checked += 1
        if meta["prompt_sha256"] != recorded:
            identity_mismatch.append((key, recorded, meta["prompt_sha256"]))
        if identity_checked >= 50:
            break

if identity_checked and identity_mismatch:
    key, expected, got = identity_mismatch[0]
    raise RuntimeError(
        f"{len(identity_mismatch)} of {identity_checked} reference prompts differ from the "
        f"ones NB 15 actually used (e.g. {key}: NB 15 {expected}, here {got}). This notebook "
        "would be measuring the sensitivity of a prompt that was never evaluated. Reconcile "
        "`evidence_block` / `build_prompt` with NB 15 before running any grid.")
if identity_checked:
    print(f"Reference prompt is byte-identical to NB 15's on {identity_checked} checked "
          "images.")
else:
    raise RuntimeError(
        f"No prompt hashes could be checked in {nb15_journal}. Run revised NB 15's E3 "
        "stage first; NB 16 cannot claim sensitivity of an unverified reference prompt.")

_demo = build_prompt(str(EVAL.index[0]), REFERENCE_VARIANT, REFERENCE_PARAPHRASE, "full")
print()
print("-" * 78)
print(_demo[0][:1200])
print("-" * 78)

## 6. The engine

One model load for the whole notebook — a single reasoner on a single fold, so there is no
reason to pay the load cost per arm. Journals and fingerprints work exactly as in NB 15: one
line per image, and the fingerprint covers the prompt text, decoding parameters and sample
count, so changing a temperature invalidates only the arms that used it.

Self-consistency draws `n` samples with distinct seeds and aggregates them with the protocol's
rule — median for mRALE and majority vote for the COVID decision. COVID probability is always
the normalized Yes/No continuation likelihood from model logits; vote share and self-reported
confidence are retained only as provenance.

In [ ]:
IMAGE_TEMPLATE_VARIANT = {}


def arm_fingerprint(arm, rows):
    prompt_identity = []
    for key in sorted(map(str, rows.index)):
        prompt, system_prompt, meta = build_prompt(
            key, arm["variant"], arm["paraphrase"], arm["depth"])
        prompt_identity.append({
            "image_key": key, "image_path": IMAGE_PATHS[key],
            "prompt_sha256": meta["prompt_sha256"],
            "system_sha256": hashlib.sha256(system_prompt.encode("utf-8")).hexdigest(),
        })
    resolved_adapter = (SPEC.get("adapter") or "").format(fold=SELECTION_FOLD) or None
    return scr.fingerprint({
        "reasoner": REASONER, "loader_spec": SPEC, "adapter": resolved_adapter,
        "revision": MODEL_REVISIONS.get(SPEC["model_id"]),
        "selection_fold": SELECTION_FOLD, "image_view": "V0_preferred_V1_fallback",
        "roster": list(ROSTER), "setting": SETTING,
        "variant": arm["variant"], "paraphrase": arm["paraphrase"], "depth": arm["depth"],
        "temperature": arm["temperature"], "top_p": arm["top_p"],
        "n_samples": arm["n_samples"], "max_new_tokens": arm["max_new_tokens"],
        "json_stop": USE_JSON_STOP_CRITERIA,
        "token_score_probe": scr.TOKEN_SCORE_PROBE_VERSION,
        "prompt_identity": scr.fingerprint(prompt_identity),
        "evaluated_rows": [item["image_key"] for item in prompt_identity],
        "system": PROMPT_VARIANTS[arm["variant"]]["system"],
        "schema": DEPTH_SCHEMA[arm["depth"]],
        "reliability": {a: {k: v for k, v in e.items() if k != "n"}
                        for a, e in sorted(RELIABILITY.items())},
        "conflict_threshold": (None if not math.isfinite(CONFLICT_THRESHOLD)
                               else round(CONFLICT_THRESHOLD, 6)),
    })


def make_arm(name, family, variant=None, paraphrase=0, depth=None, temperature=0.0,
             top_p=1.0, n_samples=1):
    depth = REFERENCE_DEPTH if depth is None else depth
    return {"name": name, "family": family,
            "variant": REFERENCE_VARIANT if variant is None else variant,
            "paraphrase": paraphrase, "depth": depth, "temperature": float(temperature),
            "top_p": float(top_p), "n_samples": int(n_samples),
            "max_new_tokens": (MAX_NEW_TOKENS_FULL_TRACE if depth == "full"
                               else MAX_NEW_TOKENS)}


def run_arm(arm, model, processor, rows=None):
    rows = EVAL if rows is None else rows
    force = FORCE_RECOMPUTE_ALL or arm["name"] in FORCE_RECOMPUTE_CONFIGS
    fingerprint = arm_fingerprint(arm, rows)
    path, _ = scr.open_journal(JOURNAL_DIR, arm["name"], fingerprint, force=force,
                               log=lambda m: print("  " + str(m).lstrip()))
    done = scr.journal_keys(path)
    pending = [k for k in map(str, rows.index) if k not in done]
    print(f"  {arm['name']}: {len(rows)} images, {len(rows) - len(pending)} cached, "
          f"{len(pending)} pending")
    if not pending:
        return path

    started = time.perf_counter()
    for index, key in enumerate(pending, start=1):
        scr.check_session_deadline(SESSION_DEADLINE, f"NB 16 {arm['name']}")
        truth = frame.loc[key]
        prompt, system_prompt, provenance = build_prompt(
            key, arm["variant"], arm["paraphrase"], arm["depth"])
        began = time.perf_counter()
        samples, raw_texts, token_counts, object_counts = [], [], [], []
        parse_error = None
        try:
            for sample_index in range(arm["n_samples"]):
                text, n_completion, n_prompt, n_objects = scr.generate_json(
                    model, processor, SPEC, IMAGE_PATHS[key], prompt, IMAGE_TEMPLATE_VARIANT,
                    system_prompt=system_prompt, max_new_tokens=arm["max_new_tokens"],
                    temperature=arm["temperature"], top_p=arm["top_p"],
                    use_json_stop=USE_JSON_STOP_CRITERIA,
                    seed=(SEED + 1000 * sample_index) if arm["temperature"] > 0 else None)
                fields, error, notes = scr.parse_reasoner_output(text, ROSTER)
                raw_texts.append(text)
                token_counts.append(n_completion)
                object_counts.append(n_objects)
                if error is None:
                    samples.append(fields)
                else:
                    parse_error = error
            if arm["n_samples"] > 1:
                fields, aggregate_error, aggregate_notes = scr.aggregate_samples(samples)
                notes = {**(notes or {}), **aggregate_notes}
                parse_error = aggregate_error
            elif samples:
                parse_error = None
            aggregate_vote_decision = fields.get("covid_pred")
            aggregate_vote_score = fields.get("covid_score")
            try:
                token_score, token_candidates = scr.covid_token_probability(
                    model, processor, SPEC, IMAGE_PATHS[key], prompt,
                    IMAGE_TEMPLATE_VARIANT, system_prompt=system_prompt)
            except Exception as scoring_exc:
                token_score, token_candidates = None, {}
                notes["covid_token_scoring_error"] = (
                    f"{type(scoring_exc).__name__}: {scoring_exc}")
            notes["covid_aggregate_vote_or_self_reported_score"] = aggregate_vote_score
            notes["covid_aggregate_vote_or_self_reported_decision"] = aggregate_vote_decision
            notes["covid_candidate_log_likelihoods"] = token_candidates
            fields["covid_score"] = token_score
            if token_score is not None:
                fields["covid_pred"] = "Yes" if token_score >= 0.5 else "No"
        except Exception as exc:
            fields, notes = {}, {}
            parse_error = f"GenerationError: {type(exc).__name__}: {exc}"
        elapsed = time.perf_counter() - began

        control_tokens = sorted({token for text in raw_texts
                                 for token in scr.control_tokens_outside_json(text)})
        row = cm.make_prediction_row(
            image_key=key, cohort="MIDRC", subcohort="MIDRC",
            filename=key.split("::")[-1], held_out_fold=int(truth["fold"]),
            agent="REASONER", arm=arm["name"], view="v0_whole", task="e6_sensitivity",
            covid_pred=fields.get("covid_pred"), covid_score=fields.get("covid_score"),
            mrale_total=fields.get("mrale_total"), mrale_right=fields.get("mrale_right"),
            mrale_left=fields.get("mrale_left"),
            extent_right=fields.get("extent_right"), density_right=fields.get("density_right"),
            extent_left=fields.get("extent_left"), density_left=fields.get("density_left"),
            gt_covid=truth.get("gt_covid"), gt_mrale_total=int(truth["gt_mrale_total"]),
            valid=parse_error is None, parse_error=parse_error,
            raw_output=(raw_texts[0][:2000] if raw_texts else ""),
            seconds=round(elapsed, 3),
            model_id=SPEC["model_id"], model_revision=MODEL_REVISIONS.get(SPEC["model_id"]),
            extra={"family": arm["family"], "variant": arm["variant"],
                   "paraphrase": arm["paraphrase"], "depth": arm["depth"],
                   "temperature": arm["temperature"], "top_p": arm["top_p"],
                   "n_samples": arm["n_samples"], "provenance": provenance, "notes": notes,
                   "completion_tokens": token_counts,
                   "total_completion_tokens": int(sum(token_counts)),
                   "json_objects_emitted": object_counts,
                   "control_tokens": control_tokens, "fingerprint": fingerprint,
                   "token_score_probe": scr.TOKEN_SCORE_PROBE_VERSION})
        cm.append_jsonl(path, row)
        if index % 25 == 0 or index == len(pending):
            rate = index / max(time.perf_counter() - started, 1e-9)
            print(f"    [{index}/{len(pending)}] {rate:.2f} img/s")
    return path


def read_arm(name):
    return cm.read_jsonl(JOURNAL_DIR / f"{name}.jsonl")


def score_arm(arm, rows=None):
    rows = read_arm(arm["name"]) if rows is None else rows
    if not rows:
        return None
    entry = OrderedDict([("arm", arm["name"]), ("family", arm["family"]),
                         ("variant", arm["variant"]), ("paraphrase", arm["paraphrase"]),
                         ("depth", arm["depth"]), ("temperature", arm["temperature"]),
                         ("top_p", arm["top_p"]), ("n_samples", arm["n_samples"]),
                         ("n", len(rows))])
    m = cm.mrale_metrics(rows)
    entry.update({"mae": round(m["mae"], 4), "rmse": round(m["rmse"], 4),
                  "qwk": round(m.get("qwk", float("nan")), 4),
                  "within1": round(m.get("within1_accuracy", float("nan")), 4),
                  "coverage": round(m.get("coverage", float("nan")), 4)})
    labelled = [r for r in rows if r.get("gt_covid") in {"Yes", "No"}]
    if labelled:
        c = cm.classification_metrics([r["gt_covid"] for r in labelled],
                                      [r.get("covid_pred") for r in labelled],
                                      [r.get("covid_score") for r in labelled])
        entry["covid_auroc"] = round(c.get("auroc", float("nan")), 4)
        entry["covid_balanced_accuracy"] = round(c.get("balanced_accuracy", float("nan")), 4)
        scored = [r for r in labelled if r.get("covid_score") is not None]
        entry["covid_score_coverage"] = round(len(scored) / len(labelled), 4)
        if scored:
            agreement = [((float(r["covid_score"]) >= 0.5) ==
                          (r.get("covid_pred") == "Yes")) for r in scored]
            entry["covid_score_decision_agreement"] = round(float(np.mean(agreement)), 4)
    o = cm.localization_free_metrics(rows)
    entry["valid_rate"] = round(o.get("output.valid_rate", float("nan")), 4)
    seconds = [r["seconds"] for r in rows if r.get("seconds") is not None]
    tokens = [(r.get("extra") or {}).get("total_completion_tokens") for r in rows]
    tokens = [t for t in tokens if t is not None]
    entry["median_seconds"] = round(float(np.median(seconds)), 3) if seconds else None
    entry["mean_completion_tokens"] = round(float(np.mean(tokens)), 1) if tokens else None
    objects = [n for r in rows for n in ((r.get("extra") or {}).get("json_objects_emitted")
                                         or [])]
    entry["mean_json_objects"] = round(float(np.mean(objects)), 3) if objects else None
    entry["n_control_token_outputs"] = sum(
        1 for r in rows if (r.get("extra") or {}).get("control_tokens"))
    return entry


all_arm_rows, arms_run = [], OrderedDict()
print("Engine ready. Journals:", JOURNAL_DIR)

## 7. E6-Q preflight — one JSON object per answer, no stray control tokens

Protocol E6-Q is a **precondition**, not a result. If the model emits its object and then
repeats it, parsing still recovers the first copy and the accuracy numbers stay valid — which
is exactly why the problem went unnoticed in the tested notebooks. But every token count,
latency figure and self-consistency result downstream would be measuring repetition.

So it is checked before the grid runs, on a small sample, and the gate blocks on it. Two
conditions: **exactly one** complete top-level JSON object per generation, no stray control
tokens outside the object, at least 0.99 strict-schema validity, and at least 0.99 coverage of
the logits-based COVID score.

In [ ]:
PREFLIGHT_N = 16
preflight_rows, e6q = [], {}

model, processor, revision = scr.load_reasoner(REASONER, SPEC, SELECTION_FOLD,
                                               MODEL_REVISIONS)
try:
    preflight_arm = make_arm("E6Q_preflight", "E6-Q")
    preflight_rows = cm.read_jsonl(run_arm(preflight_arm, model, processor,
                                           rows=EVAL.head(PREFLIGHT_N)))
    counts = [n for r in preflight_rows
              for n in ((r.get("extra") or {}).get("json_objects_emitted") or [])]
    stray = [(str(r["image_key"]), (r.get("extra") or {}).get("control_tokens"))
             for r in preflight_rows if (r.get("extra") or {}).get("control_tokens")]
    valid = [bool(r.get("valid")) for r in preflight_rows]
    scored = [r for r in preflight_rows if r.get("covid_score") is not None]
    score_agreement = [((float(r["covid_score"]) >= 0.5) ==
                        (r.get("covid_pred") == "Yes")) for r in scored]
    e6q = {"n_generations": len(counts),
           "n_exactly_one_object": int(sum(1 for n in counts if n == 1)),
           "n_zero_objects": int(sum(1 for n in counts if n == 0)),
           "n_repeated_objects": int(sum(1 for n in counts if n > 1)),
           "mean_objects": float(np.mean(counts)) if counts else float("nan"),
           "n_stray_control_tokens": len(stray),
           "valid_rate": float(np.mean(valid)) if valid else float("nan"),
           "covid_score_coverage": len(scored) / len(preflight_rows) if preflight_rows else 0.0,
           "covid_score_decision_agreement": (float(np.mean(score_agreement))
                                                if score_agreement else float("nan")),
           "template_variant": IMAGE_TEMPLATE_VARIANT.get(REASONER)}
    pd.DataFrame([e6q]).to_csv(NB16_DIR / "e6q_output_hygiene.csv", index=False)
    print()
    print(json.dumps(e6q, indent=2))
    if e6q["n_repeated_objects"]:
        print()
        print(f"E6-Q NOT RESOLVED: {e6q['n_repeated_objects']} generation(s) emitted more than")
        print("  one JSON object. Accuracy is unaffected (the first copy is parsed) but token")
        print("  counts, latency and every self-consistency figure below would be measuring")
        print("  repetition. Check USE_JSON_STOP_CRITERIA and the model's EOS handling.")
    if stray:
        print(f"  stray control tokens on {len(stray)} output(s), e.g. {stray[0]}")
    if e6q["n_zero_objects"]:
        print(f"  {e6q['n_zero_objects']} generation(s) produced no JSON object at all. If the")
        print(f"  template variant is {e6q['template_variant']!r}, check image binding first —")
        print("  a model that never saw the radiograph cannot produce a schema-shaped answer.")
except Exception as exc:
    print(f"Preflight failed: {type(exc).__name__}: {exc}")
    scr.release(model, processor)
    raise

## 8. E6-D — temperature x top-p

Greedy decoding is the primary setting; this grid exists to show the headline result is not an
artefact of one lucky configuration. Top-p is irrelevant at temperature 0 (there is no
sampling), so that row is run once rather than three times.

The number to look at is the **spread across the grid** relative to the confidence interval
NB 15 reported. A spread that fits inside the interval is the desired outcome and should be
stated as such: the conclusion does not depend on decoding.

In [ ]:
e6d_rows = []
if RUN_E6D:
    print("=" * 78)
    print("E6-D — temperature x top-p")
    print("=" * 78)
    grid = []
    for temperature in TEMPERATURES:
        if temperature == 0.0:
            grid.append(make_arm("E6D_T0.0_greedy", "E6-D", temperature=0.0, top_p=1.0))
        else:
            for top_p in TOP_P:
                grid.append(make_arm(f"E6D_T{temperature}_p{top_p}", "E6-D",
                                     temperature=temperature, top_p=top_p))
    for arm in grid:
        run_arm(arm, model, processor)
        entry = score_arm(arm)
        if entry:
            e6d_rows.append(entry)
            arms_run[arm["name"]] = arm

e6d = pd.DataFrame(e6d_rows)
if len(e6d):
    all_arm_rows.extend(e6d_rows)
    print()
    print(e6d[["arm", "temperature", "top_p", "n", "mae", "qwk", "valid_rate",
               "median_seconds", "mean_completion_tokens"]].to_string(index=False))
    spread = float(e6d["mae"].max() - e6d["mae"].min())
    greedy = e6d[e6d["temperature"] == 0.0]["mae"]
    print()
    print(f"Decoding spread: {spread:.4f} MAE across {len(e6d)} settings"
          + (f"; greedy = {float(greedy.iloc[0]):.4f}" if len(greedy) else ""))
    print("  Compare this against the fold-level CI NB 15 reported. If the spread fits inside")
    print("  it, decoding is not driving the result — which is the point of running the grid.")

## 9. E6-C — self-consistency, with its price attached

`n` samples at T=0.7, median for mRALE and majority vote for COVID. Latency and token cost
scale linearly with `n`, so the accuracy gain has to be read against a 3x or 5x compute bill.

The sample standard deviation per case is kept as well: it is a free uncertainty estimate, and
if it correlates with error it is more useful than the accuracy gain itself.

In [ ]:
e6c_rows = []
if RUN_E6C:
    print("=" * 78)
    print("E6-C — self-consistency")
    print("=" * 78)
    for n in SELF_CONSISTENCY_N:
        arm = make_arm(f"E6C_n{n}", "E6-C", temperature=SELF_CONSISTENCY_TEMPERATURE,
                       top_p=0.95, n_samples=n)
        run_arm(arm, model, processor)
        entry = score_arm(arm)
        if entry:
            rows = read_arm(arm["name"])
            sds = [(r.get("extra") or {}).get("notes", {}).get("sample_sd") for r in rows]
            sds = [s for s in sds if s is not None]
            entry["mean_sample_sd"] = round(float(np.mean(sds)), 4) if sds else None
            if sds and n > 1:
                errors, spreads = [], []
                for row in rows:
                    sd = (row.get("extra") or {}).get("notes", {}).get("sample_sd")
                    if sd is None or row.get("mrale_total") is None:
                        continue
                    errors.append(abs(float(row["mrale_total"])
                                      - float(row["gt_mrale_total"])))
                    spreads.append(float(sd))
                if len(errors) > 10 and np.std(spreads) > 1e-9:
                    from scipy.stats import spearmanr
                    entry["sd_vs_error_rho"] = round(
                        float(spearmanr(spreads, errors)[0]), 4)
            e6c_rows.append(entry)
            arms_run[arm["name"]] = arm

e6c = pd.DataFrame(e6c_rows)
if len(e6c):
    all_arm_rows.extend(e6c_rows)
    print()
    columns = [c for c in ["arm", "n_samples", "mae", "qwk", "valid_rate", "median_seconds",
                           "mean_completion_tokens", "mean_sample_sd", "sd_vs_error_rho"]
               if c in e6c.columns]
    print(e6c[columns].to_string(index=False))
    base = e6c[e6c["n_samples"] == 1]
    best = e6c.sort_values("mae").iloc[0]
    if len(base):
        gain = float(base.iloc[0]["mae"] - best["mae"])
        cost = (float(best["median_seconds"]) / float(base.iloc[0]["median_seconds"])
                if base.iloc[0]["median_seconds"] else float("nan"))
        print()
        print(f"Best self-consistency arm: {best['arm']}, {gain:+.4f} MAE for {cost:.1f}x "
              "the latency of a single sample.")
        if gain < 0.10:
            print("  That gain is inside the noise band. Self-consistency is not worth its")
            print("  cost here; report it as a negative result rather than omitting it.")
    if "sd_vs_error_rho" in e6c.columns and e6c["sd_vs_error_rho"].notna().any():
        print("  `sd_vs_error_rho` > 0 means sample disagreement predicts error, which makes")
        print("  it a usable free uncertainty estimate regardless of the accuracy gain.")

## 10. E6-P — prompt variants and brittleness

Three variants, three pre-registered paraphrases each. The variants answer *what information
the prompt needs to carry*; the paraphrases answer *how much the exact wording matters*.

**Brittleness is the standard deviation across paraphrases within a variant.** Paraphrases say
the same thing, so any spread is the system responding to wording rather than to the
radiograph. A brittleness comparable to the difference between variants means variant
comparisons are not interpretable, and the manuscript should say so instead of ranking them.

In [ ]:
e6p_rows = []
if RUN_E6P:
    print("=" * 78)
    print("E6-P — prompt variants and paraphrases")
    print("=" * 78)
    for variant in PROMPT_VARIANTS:
        for paraphrase in range(3):
            arm = make_arm(f"E6P_{variant}_p{paraphrase}", "E6-P", variant=variant,
                           paraphrase=paraphrase)
            run_arm(arm, model, processor)
            entry = score_arm(arm)
            if entry:
                e6p_rows.append(entry)
                arms_run[arm["name"]] = arm

e6p = pd.DataFrame(e6p_rows)
brittleness = pd.DataFrame()
if len(e6p):
    all_arm_rows.extend(e6p_rows)
    print()
    print(e6p[["arm", "variant", "paraphrase", "mae", "qwk", "valid_rate",
               "mean_completion_tokens"]].to_string(index=False))

    records = []
    for variant, group in e6p.groupby("variant"):
        maes = group["mae"].astype(float)
        records.append({
            "variant": variant, "n_paraphrases": len(group),
            "mae_mean": round(float(maes.mean()), 4),
            "mae_sd_across_paraphrases": round(float(maes.std(ddof=1)), 4)
            if len(maes) > 1 else None,
            "mae_min": round(float(maes.min()), 4), "mae_max": round(float(maes.max()), 4),
            "mae_range": round(float(maes.max() - maes.min()), 4),
            "valid_rate_min": round(float(group["valid_rate"].min()), 4),
        })
    brittleness = pd.DataFrame(records).sort_values("mae_mean")
    brittleness.to_csv(NB16_DIR / "prompt_brittleness.csv", index=False)
    print()
    print(brittleness.to_string(index=False))

    between = float(brittleness["mae_mean"].max() - brittleness["mae_mean"].min())
    within = float(brittleness["mae_sd_across_paraphrases"].max()) \
        if brittleness["mae_sd_across_paraphrases"].notna().any() else float("nan")
    print()
    print(f"Between-variant spread {between:.4f} MAE vs worst within-variant paraphrase SD "
          f"{within:.4f}.")
    if math.isfinite(within) and between <= 2 * within:
        print("  Paraphrase noise is comparable to the difference between variants, so the")
        print("  variants cannot be meaningfully ranked. Report the brittleness, not a winner.")
    else:
        print("  Variant differences exceed paraphrase noise, so the prompt's content — not")
        print("  its wording — is doing the work.")

## 11. E6-T — reasoning depth

`none` / `short` / `full`. The protocol is explicit that only auditable evidence summaries are
published; private chain-of-thought is not reproduced in the paper. What is reported here is
accuracy, output token count, latency and JSON validity per depth.

The `full` arm is the one most likely to break JSON validity: a model asked to reason at length
inside a JSON object frequently runs out of tokens mid-string. That failure shows up as a
falling `valid_rate`, not as a falling MAE, which is why both are in the table.

In [ ]:
e6t_rows = []
if RUN_E6T:
    print("=" * 78)
    print("E6-T — reasoning depth")
    print("=" * 78)
    for depth in DEPTHS:
        arm = make_arm(f"E6T_{depth}", "E6-T", depth=depth)
        run_arm(arm, model, processor)
        entry = score_arm(arm)
        if entry:
            rows = read_arm(arm["name"])
            truncated = sum(1 for r in rows
                            if r.get("parse_error") and "Expecting" in str(r["parse_error"]))
            entry["n_likely_truncated"] = truncated
            e6t_rows.append(entry)
            arms_run[arm["name"]] = arm

e6t = pd.DataFrame(e6t_rows)
if len(e6t):
    all_arm_rows.extend(e6t_rows)
    print()
    columns = [c for c in ["arm", "depth", "mae", "qwk", "valid_rate",
                           "mean_completion_tokens", "median_seconds", "n_likely_truncated"]
               if c in e6t.columns]
    print(e6t[columns].to_string(index=False))
    if len(e6t) > 1:
        best, worst = e6t.sort_values("mae").iloc[0], e6t.sort_values("mae").iloc[-1]
        tokens = e6t.set_index("depth")["mean_completion_tokens"].to_dict()
        print()
        print(f"Depth spread: {float(worst['mae'] - best['mae']):.4f} MAE; token cost by "
              f"depth {tokens}")
        if float(worst["mae"] - best["mae"]) < 0.10:
            print("  Reasoning depth does not measurably change accuracy on this task. The")
            print("  rationale is worth keeping for auditability, not for accuracy — and that")
            print("  is the honest framing for the manuscript.")

# Everything the model was released after this point.
scr.release(model, processor)
print()
print("Model released.")

## 12. Latency against accuracy

Every arm on one axis pair. This is the table that stops a 5x-compute arm being reported as an
improvement because it moved MAE by 0.05.

The Pareto frontier is computed explicitly: an arm is on it when no other arm is both faster
and more accurate. Arms off the frontier are dominated and should not be recommended,
whatever their raw MAE.

In [ ]:
latency = pd.DataFrame(all_arm_rows)
if len(latency):
    latency = latency.sort_values("mae")
    usable = latency.dropna(subset=["median_seconds", "mae"])
    frontier = []
    for _, row in usable.iterrows():
        dominated = ((usable["mae"] <= row["mae"]) & (usable["median_seconds"]
                                                      <= row["median_seconds"])
                     & ((usable["mae"] < row["mae"])
                        | (usable["median_seconds"] < row["median_seconds"]))).any()
        frontier.append(not dominated)
    usable = usable.assign(pareto_frontier=frontier)
    latency = latency.merge(usable[["arm", "pareto_frontier"]], on="arm", how="left")
    latency.to_csv(NB16_DIR / "latency_vs_accuracy.csv", index=False)
    pd.DataFrame(all_arm_rows).to_csv(NB16_DIR / "e6_sensitivity_grid.csv", index=False)

    columns = [c for c in ["arm", "family", "mae", "valid_rate", "median_seconds",
                           "mean_completion_tokens", "pareto_frontier"] if c in latency.columns]
    print(latency[columns].to_string(index=False))
    print()
    on_frontier = latency[latency["pareto_frontier"] == True]["arm"].tolist()
    print(f"Pareto frontier ({len(on_frontier)} arms): {on_frontier}")
    print("  Arms off the frontier are strictly worse on both axes and should not be")
    print("  recommended regardless of their headline MAE.")
else:
    print("No arms were run.")

## 13. Run configuration and gate

E6 is exploratory, so most of what it produces is a warning that shapes the manuscript rather
than a blocking condition. Two things do block:

1. **E6-Q must be resolved** — exactly one strict-schema JSON object per generation, no stray
   control tokens, and usable logits-based COVID scores. Everything in the
   latency and token columns is meaningless otherwise.
2. **The reference arm must reproduce NB 15** — same prompt, and an MAE consistent with what
   NB 15 reported on the same rows. If the greedy reference arm here disagrees with NB 15's
   E3 arm on the same images, one of the two notebooks is not running the system it claims to.

In [ ]:
failures, warnings = [], []

# ---- Gate 1: E6-Q -------------------------------------------------------------------------
if e6q:
    if e6q.get("n_repeated_objects"):
        failures.append(
            f"E6-Q unresolved: {e6q['n_repeated_objects']} of {e6q['n_generations']} "
            "generations emitted more than one JSON object. Accuracy is unaffected but every "
            "token-count, latency and self-consistency figure in this notebook would be "
            "measuring repetition. Fix stop criteria / EOS handling and re-run.")
    if e6q.get("n_stray_control_tokens"):
        failures.append(f"E6-Q unresolved: {e6q['n_stray_control_tokens']} output(s) "
                        "carried stray control tokens outside the JSON object.")
    if e6q.get("n_zero_objects"):
        failures.append(
            f"{e6q['n_zero_objects']} preflight generation(s) produced no JSON object "
            f"(template variant {e6q.get('template_variant')!r}). Check image binding before "
            "reading anything below.")
    if e6q.get("valid_rate", 0.0) < 0.99:
        failures.append(f"E6-Q strict-schema valid rate is {e6q.get('valid_rate', 0):.1%}; "
                        "at least 99% is required before the sensitivity grid is usable.")
    if e6q.get("covid_score_coverage", 0.0) < 0.99:
        failures.append(f"COVID logits-score coverage is "
                        f"{e6q.get('covid_score_coverage', 0):.1%}; at least 99% required.")
    agreement = e6q.get("covid_score_decision_agreement")
    if agreement is None or not math.isfinite(float(agreement)) or float(agreement) < 0.995:
        failures.append(f"COVID score/decision agreement is {agreement}; at least 99.5% "
                        "required by protocol 7.2.")
else:
    failures.append("The E6-Q preflight did not run, so output hygiene is unverified.")

# ---- Gate 2: does the reference arm reproduce NB 15? ---------------------------------------
reference_arm = "E6D_T0.0_greedy"
reference_rows = read_arm(reference_arm)
nb15_reference = cm.read_jsonl(NB15_DIR / "journals"
                               / f"E3_{SETTING}__fold{SELECTION_FOLD}.jsonl")
if reference_rows and nb15_reference:
    here = {str(r["image_key"]): r for r in reference_rows}
    there = {str(r["image_key"]): r for r in nb15_reference}
    shared = sorted(set(here) & set(there))
    if len(shared) >= 20:
        mae_here = cm.mrale_metrics([here[k] for k in shared])["mae"]
        mae_there = cm.mrale_metrics([there[k] for k in shared])["mae"]
        agree = sum(1 for k in shared
                    if here[k].get("mrale_total") == there[k].get("mrale_total"))
        agreement = agree / len(shared)
        print(f"Reference arm vs NB 15 on {len(shared)} shared images: "
              f"MAE {mae_here:.4f} here vs {mae_there:.4f} there; "
              f"identical answers on {agreement:.1%}")
        if agreement < 0.95:
            failures.append(
                f"The greedy reference arm reproduces only {agreement:.1%} of NB 15's answers "
                f"on the same {len(shared)} images (MAE {mae_here:.4f} vs {mae_there:.4f}). "
                "Both runs are greedy on an identical prompt, so they should agree almost "
                "exactly. One of the notebooks is not running the system it claims to — check "
                "the adapter path, the message schema, and the dtype before trusting either.")
    else:
        failures.append(f"Only {len(shared)} images are shared with NB 15's E3 arm; at "
                        "least 20 are required to verify reference-arm reproduction.")
else:
    failures.append("Could not compare the reference arm against NB 15; run NB 15's E3 stage.")

if identity_checked == 0:
    failures.append("The reference prompt was never verified byte-for-byte against NB 15.")

# ---- Warnings that shape the manuscript ---------------------------------------------------
if len(e6d) > 1:
    spread = float(e6d["mae"].max() - e6d["mae"].min())
    warnings.append(
        f"E6-D decoding spread {spread:.4f} MAE across {len(e6d)} settings. "
        + ("Decoding does not drive the result." if spread < 0.30 else
           "Decoding moves the result appreciably; the primary greedy setting must be "
           "reported as a choice, with this grid beside it."))

if len(e6c) > 1 and "median_seconds" in e6c.columns:
    base = e6c[e6c["n_samples"] == 1]
    best = e6c.sort_values("mae").iloc[0]
    if len(base):
        gain = float(base.iloc[0]["mae"] - best["mae"])
        warnings.append(
            f"E6-C best gain {gain:+.4f} MAE at n={int(best['n_samples'])} "
            f"({float(best['median_seconds']):.2f}s vs "
            f"{float(base.iloc[0]['median_seconds']):.2f}s per case). "
            + ("Not worth the compute; report as a negative result." if gain < 0.10 else
               "Worth reporting, with the latency cost stated."))

if len(brittleness) and brittleness["mae_sd_across_paraphrases"].notna().any():
    worst = float(brittleness["mae_sd_across_paraphrases"].max())
    warnings.append(
        f"Prompt brittleness: worst within-variant paraphrase SD {worst:.4f} MAE. "
        + ("Small relative to the reported effects." if worst < 0.20 else
           "Large enough that prompt wording is a material source of variance and must be "
           "declared in the limitations."))

if len(e6t) > 1:
    spread = float(e6t["mae"].max() - e6t["mae"].min())
    validity = e6t.set_index("depth")["valid_rate"].to_dict()
    warnings.append(f"E6-T depth spread {spread:.4f} MAE; valid-JSON rate by depth {validity}. "
                    + ("Rationales are worth keeping for auditability, not accuracy."
                       if spread < 0.10 else "Reasoning depth changes accuracy materially."))

low_validity = [row["arm"] for row in all_arm_rows
                if row.get("valid_rate") is not None and row["valid_rate"] < 0.95]
if low_validity:
    warnings.append(f"{len(low_validity)} exploratory arm(s) fell below a 0.95 valid-JSON "
                    f"rate: {low_validity[:6]}. Their metrics carry the invalid-output "
                    "penalty and should not be compared to confirmatory arms.")

cm.write_json(NB16_DIR / "run_config.json", {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "16_decoding_prompt_and_depth_sensitivity.ipynb",
    "protocol_experiments": ["E6-D", "E6-C", "E6-P", "E6-T", "E6-Q"],
    "status": "EXPLORATORY — not a confirmatory family, no multiplicity correction applies",
    "seed": SEED,
    "inherited_from_nb15": {"reasoner": REASONER, "setting": SETTING, "roster": ROSTER,
                             "reasoner_spec": recorded_spec},
    "evaluation_rows": int(len(EVAL)),
    "evaluation_scope": (f"evaluation half of fold {SELECTION_FOLD}'s inner-validation pool — "
                         "the same rows NB 15 selected on; no test fold is touched"),
    "reference_prompt_identity_checked": identity_checked,
    "n_arms": len(all_arm_rows),
    "image_template_variant": IMAGE_TEMPLATE_VARIANT,
    "image_view": "V0 preferred, V1 thorax fallback only when V0 is unavailable",
    "covid_score_source": scr.TOKEN_SCORE_PROBE_VERSION,
    "e6q": e6q,
})


def report(title, messages):
    print(title)
    for message in messages or []:
        print("  -", message)
    if not messages:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)
cm.write_json(NB16_DIR / "gate_nb16.json",
              {"passed": not failures, "failures": failures, "warnings": warnings})
if failures:
    detail = "\n".join(f"  [{i + 1}] {m}" for i, m in enumerate(failures))
    raise AssertionError(f"NB 16 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 16 gate: PASSED")
print()
print(f"{len(all_arm_rows)} exploratory arms on {len(EVAL)} images. Stage C is complete; "
      "NB 17 (Stage D) is the single source of truth for intervals and p-values.")